# SAC continuous-action policy smoke

Build one untrained SAC policy on a tiny synthetic `TradingEnv` and sample one action. This checks basic wiring only; it is neither training nor a trading result.


In [ ]:
import numpy as np
import pandas as pd
from omegaconf import OmegaConf

from learning.rl.agent import build_agent
from simulation.environment.env import TradingEnv

rng = np.random.default_rng(42)
n = 120
index = pd.date_range("2020-01-01", periods=n, freq="1min")
close = 100 + np.cumsum(rng.normal(0, 0.1, n))
bars = pd.DataFrame(
    {
        "open": close,
        "high": close + 0.2,
        "low": close - 0.2,
        "close": close,
        "volume": 1000,
        "session_id": 0,
    },
    index=index,
)
features = pd.DataFrame({"signal": rng.normal(size=n)}, index=index)
env = TradingEnv(bars, features, obs_window=10, episodic=True, continuous_actions=True)
cfg = OmegaConf.create(
    {
        "env": {"obs_window": 10},
        "ppo": {"batch_size": 32},
        "sac": {"buffer_size": 5000, "batch_size": 16, "learning_starts": 200},
    }
)
model = build_agent(env, cfg, arch="tcn", algo="sac", device="cpu", seed=7)

In [ ]:
obs = model.get_env().reset()
action, _ = model.predict(obs, deterministic=True)
print("action:", action)
assert action.shape == (1, 1)
assert np.all(action >= -1.0) and np.all(action <= 1.0)

SAC requires a continuous `Box` action space here. This notebook does not call `.learn()`; see the [SAC guide](../../../concepts/sac.md) for algorithm constraints.
